In [10]:
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt
import pandas as pd

In [26]:
class ProbabilityNN(nn.Module):
    def __init__(self, n_input, layers):
        # Layers is a list of ints where each integer describes the dimension of the hidden layers
        super().__init__()
        self.layer_functions = []
        self.layer_functions.append(nn.Linear(n_input, layers[0]))
        self.layer_functions.append(nn.ReLU())
        for i in range(len(layers)):
            if i == 0:
                continue
            self.layer_functions.append(nn.Linear(layers[i - 1], layers[i]))
            self.layer_functions.append(nn.ReLU())
        self.layer_functions.append(nn.Linear(layers[-1], 1))
        self.layer_functions.append(nn.Sigmoid())

        self.layer_functions = nn.Sequential(*self.layer_functions)

    def forward(self, x):
        return self.layer_functions(x)

class Data(Dataset):
    def __init__(self, features, targets, n_samples = 1000):
        self.features = features
        self.targets = targets
    
    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx], self.targets[idx]

In [28]:
features = torch.randn(1000, 3)
targets = torch.rand(1000)

dataset = Data(features, targets)

layers = [16, 16, 16]

model = ProbabilityNN(3, layers)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr = 0.0001)

# Begin training with Epochs
num_epochs = 5
model.train()

data_loader = DataLoader(dataset, batch_size = 32, shuffle = True)

for epoch in range(num_epochs):
    epoch_loss = 0.0
    for batch_features, batch_targets in data_loader:
        optimizer.zero_grad()
        output = model(batch_features)
        loss = criterion(output, batch_targets.view(-1, 1))
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        #print(f"{epoch} Epoch Loss: {loss.item()}")
    avg_loss = epoch_loss / len(data_loader)
    print(f"Epoch [{epoch+1}/{num_epochs}], Average Loss: {avg_loss:.4f}")


Epoch [1/5], Average Loss: 0.6999
Epoch [2/5], Average Loss: 0.6989
Epoch [3/5], Average Loss: 0.6982
Epoch [4/5], Average Loss: 0.6972
Epoch [5/5], Average Loss: 0.6970
